In [1]:
# --- Spark Bootstrap (Colab-safe) ---
# Java 17 + PySpark 3.5.x (works together)
!apt -qq update -y > /dev/null
!apt -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q install pyspark==3.5.1

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"   # avoid hostname resolution issues

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("colab-spark")
    # bind driver explicitly; prevents py4j 'connection refused'
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    # modest settings for Colab RAM
    .config("spark.sql.shuffle.partitions", "48")
    .config("spark.driver.memory", "12g")
    .config("spark.driver.maxResultSize", "1g")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark up ✔", spark.version, "| JAVA_HOME =", os.environ["JAVA_HOME"])




W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Spark up ✔ 3.5.1 | JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64


In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length

# Initialize Spark session (assuming it is already created)
spark = SparkSession.builder.appName("BooksLlama").getOrCreate()

# Load dataset into pandas DataFrame
ds = load_dataset("open-phi/programming_books_llama", split="train")
df_pd = ds.to_pandas()

# Convert ndarray fields to list (for Spark compatibility)
columns_to_convert = ['outline', 'concepts', 'queries', 'context']

def convert_ndarray_to_list(row):
    if isinstance(row, np.ndarray):
        return row.tolist()
    return row

for col_name in columns_to_convert:
    if col_name in df_pd.columns:
        df_pd[col_name] = df_pd[col_name].apply(convert_ndarray_to_list)

# Convert to Spark DataFrame with all columns cast to string
def df_to_spark_string(df_chunk):
    for col in df_chunk.columns:
        df_chunk[col] = df_chunk[col].astype(str)
    return spark.createDataFrame(df_chunk)

chunk_size = 5000
chunks = [df_pd.iloc[i:i + chunk_size] for i in range(0, len(df_pd), chunk_size)]

spark_dfs = []
for idx, chunk in enumerate(chunks):
    print(f"Processing chunk {idx + 1}/{len(chunks)}")
    sdf = df_to_spark_string(chunk)
    spark_dfs.append(sdf)

# Union all chunks
df_spark = spark_dfs[0]
for sdf in spark_dfs[1:]:
    df_spark = df_spark.union(sdf)

print(f"Total rows loaded in Spark DataFrame: {df_spark.count()}")

# Data cleaning by dropping rows with nulls in critical columns available
required_columns = ['markdown', 'model']  # Adjust based on use case
df_clean = df_spark.dropna(subset=required_columns)

# Add content length column on 'markdown' which seems closest to content
df_clean = df_clean.withColumn("content_length", length(col("markdown")))

# Train/test split
train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Train set size: {train_df.count()}, Test set size: {test_df.count()}")

# Show sample cleaned data
df_clean.select("model", "content_length", "markdown").show(5)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing chunk 1/23


/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chunk[col] = df_chunk[col].astype(str)
/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chunk[col] = df_chunk[col].astype(str)
/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/sta

Processing chunk 2/23


/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chunk[col] = df_chunk[col].astype(str)
/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chunk[col] = df_chunk[col].astype(str)
/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/sta

...

Processing chunk 22/23


/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chunk[col] = df_chunk[col].astype(str)
/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chunk[col] = df_chunk[col].astype(str)
/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/sta

Processing chunk 23/23


/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chunk[col] = df_chunk[col].astype(str)
/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_chunk[col] = df_chunk[col].astype(str)
/tmp/ipython-input-674062296.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/sta

Total rows loaded in Spark DataFrame: 111048
Train set size: 88941, Test set size: 22107
+-------+--------------+--------------------+
|  model|content_length|            markdown|
+-------+--------------+--------------------+
|gpt-3.5|         35261|\n\n# Understandi...|
|gpt-3.5|         32382|\n\n# Understandi...|
|gpt-3.5|         22800|\n\n# The role of...|
|gpt-3.5|         18774|\n\n# Common data...|
|gpt-3.5|         30255|\n\n# Understandi...|
+-------+--------------+--------------------+
only showing top 5 rows

